In [ ]:
import numpy as np
import random
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


print("Loading dataset...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

# Normalize
X = X / 255.0
y = y.astype(int)

# Reduce dataset size
X = X[:5000]
y = y[:5000]

num_labeled = 100

indices = np.arange(len(X))
np.random.shuffle(indices)

labeled_idx = indices[:num_labeled]
unlabeled_idx = indices[num_labeled:]

X_labeled = X[labeled_idx]
y_labeled = y[labeled_idx]

X_unlabeled = X[unlabeled_idx]
y_unlabeled_true = y[unlabeled_idx]  # for evaluation

print(f"Labeled: {len(X_labeled)}, Unlabeled: {len(X_unlabeled)}")

print("\nRunning Self-Training...")

# Combine labeled + unlabeled (unlabeled = -1)
y_semi = np.copy(y)
y_semi[unlabeled_idx] = -1

base_clf = GaussianNB()
self_training_model = SelfTrainingClassifier(base_clf)

self_training_model.fit(X, y_semi)

y_pred_self = self_training_model.predict(X_unlabeled)

acc_self = accuracy_score(y_unlabeled_true, y_pred_self)
print("Self-Training Accuracy:", acc_self)
print("\nRunning Co-Training...")

# Split features into two views
X1 = X[:, :392]
X2 = X[:, 392:]

X1_labeled = X1[labeled_idx]
X2_labeled = X2[labeled_idx]

X1_unlabeled = X1[unlabeled_idx]
X2_unlabeled = X2[unlabeled_idx]

# Two classifiers
clf1 = DecisionTreeClassifier()
clf2 = LogisticRegression(max_iter=1000)

# Train initial models
clf1.fit(X1_labeled, y_labeled)
clf2.fit(X2_labeled, y_labeled)

# -----------------------------
# Threshold-based Co-Training
# -----------------------------
threshold = 0.9

# Get probabilities
proba1 = clf1.predict_proba(X1_unlabeled)
proba2 = clf2.predict_proba(X2_unlabeled)

# Confidence scores
conf1 = np.max(proba1, axis=1)
conf2 = np.max(proba2, axis=1)

# Predicted labels
pred1 = np.argmax(proba1, axis=1)
pred2 = np.argmax(proba2, axis=1)

# Get indices of high-confidence samples
idx1 = np.where(conf1 >= threshold)[0]
idx2 = np.where(conf2 >= threshold)[0]

print(f"Model1 confident samples: {len(idx1)}")
print(f"Model2 confident samples: {len(idx2)}")

# -----------------------------
# Proper Cross-View Mapping
# -----------------------------

# Model 1 confident samples (used to teach Model 2)
X2_from_1 = X2_unlabeled[idx1]
y1_high = pred1[idx1]

# Model 2 confident samples (used to teach Model 1)
X1_from_2 = X1_unlabeled[idx2]
y2_high = pred2[idx2]

# -----------------------------
# Retrain (Cross Teaching)
# -----------------------------

# Model 1 teaches Model 2
clf2.fit(
    np.vstack((X2_labeled, X2_from_1)),
    np.hstack((y_labeled, y1_high))
)

# Model 2 teaches Model 1
clf1.fit(
    np.vstack((X1_labeled, X1_from_2)),
    np.hstack((y_labeled, y2_high))
)

# -----------------------------
# Evaluation
# -----------------------------
y_pred_cotrain = clf1.predict(X1_unlabeled)

acc_cotrain = accuracy_score(y_unlabeled_true, y_pred_cotrain)
print("Co-Training Accuracy (Threshold):", acc_cotrain)
# -----------------------------
# CLUSTERING (K-Means)
# -----------------------------
print("\nRunning K-Means Clustering...")

kmeans = KMeans(n_clusters=10, random_state=42)
clusters = kmeans.fit_predict(X)

# Assign labels to clusters
cluster_labels = {}

for i in range(10):
    cluster_points = labeled_idx[clusters[labeled_idx] == i]
   
    if len(cluster_points) == 0:
        continue
   
    labels = y[cluster_points]
    majority_label = np.bincount(labels).argmax()
    cluster_labels[i] = majority_label

# Predict for unlabeled
y_pred_cluster = []

for idx in unlabeled_idx:
    cluster_id = clusters[idx]
    if cluster_id in cluster_labels:
        y_pred_cluster.append(cluster_labels[cluster_id])
    else:
        y_pred_cluster.append(random.choice(range(10)))

y_pred_cluster = np.array(y_pred_cluster)

acc_cluster = accuracy_score(y_unlabeled_true, y_pred_cluster)
print("Clustering Accuracy:", acc_cluster)

# -----------------------------
# Summary
# -----------------------------
print("\n===== FINAL COMPARISON =====")
print(f"Self-Training Accuracy : {acc_self:.4f}")
print(f"Co-Training Accuracy   : {acc_cotrain:.4f}")
print(f"K-Means Accuracy       : {acc_cluster:.4f}")

Loading dataset...
Labeled: 100, Unlabeled: 4900

Running Self-Training...
Self-Training Accuracy: 0.5408163265306123

Running Co-Training...
Model1 confident samples: 4900
Model2 confident samples: 1214
Co-Training Accuracy (Threshold): 0.546530612244898

Running K-Means Clustering...
Clustering Accuracy: 0.5118367346938776

===== FINAL COMPARISON =====
Self-Training Accuracy : 0.5408
Co-Training Accuracy   : 0.5465
K-Means Accuracy       : 0.5118
